In [1]:
import os
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader, Subset,random_split
import numpy as np
from PIL import Image
import lightning as L
import torch.nn as nn
from model_student import AEStudent
from model_teacher import ResNet18Segmentation
from rfidataset import NenuFARDatasetH5
from torchmetrics.classification import MulticlassAccuracy
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
from pathlib import Path
from torchvision import datasets, transforms
from torchvision.transforms import v2
from torchvision import tv_tensors
from torchmetrics.classification import BinaryJaccardIndex, BinaryF1Score
import torch.nn.functional as F




In [ ]:
h5_path = "teacher_model/model_and_data/train_supervised_secondattempt.h5"

full_dataset = NenuFARDatasetH5(
    h5_path=h5_path,
    transform=None,
    return_meta=True,
)



#Réduction du dataset pour tester
TEST = False         # mettre a False pour l'entrainement complet
TEST_SIZE = 200       # nombre de données a garder en mode test

if TEST:
    print("-----Mode test-----")
    n_total_full = len(full_dataset)
    rng = np.random.default_rng(seed=42)
    debug_indices = rng.choice(n_total_full, size=min(TEST_SIZE, n_total_full), replace=False)
    dataset = Subset(full_dataset, debug_indices)
else:
    dataset = full_dataset


n_total = len(dataset)                
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_subset, val_subset, test_subset = random_split(
    dataset,                          
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_subset, batch_size=4, shuffle=False, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=4, shuffle=False, num_workers=0)
test_loader = DataLoader(test_subset, batch_size=4, shuffle=False, num_workers=0)

print(f"Dataset utilise : {n_total} echantillons "
      f"(train={n_train}, val={n_val}, test={n_test})")


Nb samples: 3000
mode test
Dataset utilise : 200 echantillons (train=140, val=30, test=30)


In [ ]:



def distillation_kl_loss(student_logits, teacher_logits, T =4.0):
    #KL divergence pixel par pixel entre student et teacher, avec temperature.
   
    B, C, H, W = student_logits.shape

    s = student_logits.permute(0, 2, 3, 1).reshape(-1, C)
    t = teacher_logits.permute(0, 2, 3, 1).reshape(-1, C)

    log_p_student = F.log_softmax(s / T, dim=1)
    p_teacher = F.softmax(t / T, dim=1)

    kl = F.kl_div(log_p_student, p_teacher, reduction="batchmean")
    return kl * (T** 2)


def compute_loss(student, teacher, x, y, alpha=0.5, T=4.0):
    student_logits = student(x)

    with torch.no_grad():
        teacher_logits = teacher(x)

    ce = F.cross_entropy(student_logits, y)
    kl = distillation_kl_loss(student_logits, teacher_logits, T)

    loss = (1 - alpha) * ce + alpha * kl
    return loss, ce, kl, student_logits


def train_one_epoch(student, teacher, loader, optimizer, device, alpha=0.5, T =4.0):
    student.train()
    teacher.eval()
 
    total_loss = 0.0
    total_acc = 0.0
 
    for x, y, _meta in loader:
        x, y = x.to(device), y.to(device)
 
        optimizer.zero_grad()
        loss, ce, kl, preds = compute_loss(student, teacher, x, y, alpha, T)
        loss.backward()
        optimizer.step()
 
        acc = (preds.argmax(dim=1) == y).float().mean()
        total_loss += loss.item()
        total_acc += acc.item()
 
    n = len(loader)
    return total_loss / n, total_acc / n
 
 
@torch.no_grad()
def evaluate(student, teacher, loader, device, alpha=0.5, T=4.0):
    student.eval()
    teacher.eval()
 
    total_loss = 0.0
    total_acc = 0.0
 
    for x, y, _meta in loader:
        x, y = x.to(device), y.to(device)
        loss, ce, kl, preds = compute_loss(student, teacher, x, y, alpha, T)
 
        acc = (preds.argmax(dim=1) == y).float().mean()
        total_loss += loss.item()
        total_acc += acc.item()
 
    n = len(loader)
    return total_loss / n, total_acc / n
 


In [ ]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Teacher
ckpt_path = "teacher_model/model_and_data/best_model.pth"  
teacher = ResNet18Segmentation(num_classes=2,pretrained=False,dropout_p=0.3).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
teacher.load_state_dict(ckpt["model_state_dict"])
teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

# Student
student = AEStudent(num_classes=2, dropout_p=0.2).to(device)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

x, y, _meta = next(iter(train_loader))
x = x.to(device)

# Vérification résolutions :
#student.eval()
#teacher.eval()
#with torch.no_grad():
#    s_logits = student(x)
#    t_logits = teacher(x)

#print("student :", s_logits.shape)
#print("teacher :", t_logits.shape)





/home/rzouhhad/miniforge3/lib/python3.13/site-packages/torch/cuda/__init__.py:1074: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [ ]:
n_epochs = 50
alpha = 0.5
T = 4.0

for epoch in range(n_epochs):
    train_loss, train_acc = train_one_epoch(
        student, teacher, train_loader, optimizer, device, alpha, T)
    val_loss, val_acc = evaluate(
        student, teacher, val_loader, device, alpha, T)

    print(f"Epoch {epoch+1}/{n_epochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

torch.save(student.state_dict(), "student_model/student.pt")

Epoch 1/2 | train_loss=0.4722 train_acc=0.9530 | val_loss=0.3930 val_acc=0.9735
Epoch 2/2 | train_loss=0.2884 train_acc=0.9749 | val_loss=0.4625 val_acc=0.9663
